In [67]:
from langchain_chroma import Chroma
import os
import chromadb
import pathlib
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [68]:
# Source 1 - the PDF. PyPDFLoader returns one document per page.

pdf_docs = PyPDFLoader("victor_technical_report.pdf").load()

# # Source 2 - the text file. Plain Python can read this on its own.
# with open("business_registration_guide.txt", "r", encoding="utf-8") as f:
#     txt_text = f.read()

print("Pages loaded from the PDF :", len(pdf_docs))
# print("Characters read from TXT  :", len(txt_text))

print()
print("First 300 characters of the PDF:")
print(pdf_docs[0].page_content[:300])

# print()
# print("First 300 characters of the TXT:")
# print(txt_text[:300])

Pages loaded from the PDF : 21

First 300 characters of the PDF:
TECHNICAL REPORT ON STUDENT INDUSTRIAL WORK 
EXPERIENCE SCHEME 
[SIWES] 
BY 
OGUNGBESAN VICTOR AYODEJI 
MATRICULATION NUMBER - 210504002 
DEPARTMENT OF MECHANICAL ENGINEERING 
FACULTY OF ENGINEERING 
EKITI STATE UNIVERSITY, ADO EKITI, EKITI STATE. 
AT 
PETIRABLE ELECTRIC POLES LTD 
KM 2 GBERIGBE – I


In [69]:
# The same splitter is used for both sources
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

# The PDF pages are already documents, so we split the documents
pdf_chunks = splitter.split_documents(pdf_docs)

# The TXT is one long plain string, so we create documents from it
# txt_chunks = splitter.create_documents([txt_text])

print("Chunks from the PDF :", len(pdf_chunks))
# print("Chunks from the TXT :", len(txt_chunks))


# ---------------------------------------------------------------
# CHOOSE WHICH SOURCE TO EMBED
# Uncomment the line you want and comment out the others.
# ---------------------------------------------------------------
chosen_chunks = pdf_chunks #+ txt_chunks     # use both together
# chosen_chunks = pdf_chunks                # use only the PDF
# chosen_chunks = txt_chunks                # use only the text file


# Chroma only needs the plain text of each chunk
documents = [chunk.page_content for chunk in chosen_chunks]

print()
print("Documents ready to embed:", len(documents))

Chunks from the PDF : 38

Documents ready to embed: 38


In [70]:
#use this to comfirm the number documents to embed 
print(f"Total Documents to be embedded {len(documents)}\n")

Total Documents to be embedded 38



In [71]:
load_dotenv()

# ---------------------------------------------------------------
# PROVIDER CONFIG
# The API keys are secret, so they live in the .env file.
# Everything else is written out here so you can see it.
#
# NOTE: the chat model and the embedding model run on TWO DIFFERENT
# servers. Each one needs its own base URL and its own key.
# Do not mix them up.
# ---------------------------------------------------------------

# --- Chat model ---
api_key         = os.getenv("OPENAI_API_KEY")
base_url        = "https://museglimmer30b.publicaai.com/v1"
chat_model_name = "meta-models/Muse-Glimmer-30B"

# --- Embedding model (different server, different key) ---
embedding_api_key    = os.getenv("EMBEDDING_API_KEY")
embedding_base_url   = "https://qwen-embed.publicaai.com/v1"
embedding_model_name = "Qwen/Qwen3-Embedding-0.6B"

print("Chat base URL     :", base_url)
print("Chat model        :", chat_model_name)
print("Embed base URL    :", embedding_base_url)
print("Embedding model   :", embedding_model_name)

Chat base URL     : https://museglimmer30b.publicaai.com/v1
Chat model        : meta-models/Muse-Glimmer-30B
Embed base URL    : https://qwen-embed.publicaai.com/v1
Embedding model   : Qwen/Qwen3-Embedding-0.6B


In [72]:
chroma_path = "siwes_report_store2"


In [73]:
# Initialize the embedding model
embeddings = OpenAIEmbeddings(
    model=embedding_model_name,
    api_key=embedding_api_key,
    base_url=embedding_base_url

)

In [74]:
# Initialize ChromaDB
siwes_report2_vdb = Chroma(
    collection_name="siwes_report2",
    embedding_function=embeddings,
    persist_directory=chroma_path
)


In [75]:
# Add documents to the vector database
siwes_report2_vdb.add_texts(
    texts=documents
)

['bb8e86f1-b0be-4e58-a93c-cfdc57ec5207',
 'c6d8f539-c845-4a39-a481-ece252ca70bc',
 '87f8a6ca-0587-40e1-b2fd-93167345eb8e',
 'cc5ef97b-31bd-46ad-9c9c-1f7153b65bb5',
 '06aa6e94-0ff4-4c59-90f2-f864fdbf95d5',
 'abf89e19-90ef-4e49-964f-0fc9b74d4db0',
 '9a4f8872-1251-4e3c-b274-f0fded1f13d4',
 '2ad217e0-930d-45a3-b80b-75d7331674cb',
 'fac0ed4a-d83e-4891-b610-cbaed051e5e9',
 '1e84b67d-d9d6-4bdb-9beb-08692f54b816',
 '3ff91aa6-84e0-4816-838c-2df2e772de12',
 '7b5c7d80-f71f-492b-8a72-8eaaaa10b842',
 '24f12020-128c-42bf-867d-d9ad9faf1a67',
 '58dcff90-0bb0-4914-a39f-85f2aff9bf12',
 '90b36303-74fb-4223-b7d1-88730f25a6f4',
 'ea79912c-d24a-4a6c-9cdb-0823269815c2',
 '07110871-9b2b-4706-aac4-d62e702023ae',
 'f9c2da36-e48b-455b-9fb1-a3eac3d585a4',
 'a8153b34-665e-484a-bd96-8fb2a33c30a6',
 '0a2dfcb4-52d7-4f0e-9d2e-cbdfbe19842d',
 '1a9ca6c1-ede8-46f1-bd8c-cf028cae53c2',
 '1cdd5e4f-b448-4824-a117-fa82774bee89',
 'dda9a3ca-8df4-4a6a-aa1d-a7ddae8e886a',
 'a6667b58-4ed0-4956-aa0c-94ad2c91379c',
 '544671f3-f0f0-

In [76]:
# Maximal Marginal Relevance (MMR) for diverse and relevant results.
siwes_report2_retriever = siwes_report2_vdb.as_retriever(search_type="mmr")

In [77]:
siwes_report2_retriever = siwes_report2_vdb.as_retriever(search_type="mmr", search_kwargs={'k': 4, 'fetch_k': 10})

In [78]:
question = "What is the name of the company where siwes was undertaken?"
siwes_report2_retriever.invoke(question)             

[Document(id='dbd5e6fd-098d-45d9-9ca8-e46bf19fa317', metadata={}, page_content='5 \n \n2.3 PETIRABLE SERVICES \nPertirable offers 3 distinct services that has served over 200 clients which includes \nindividuals and communities, government agencies and multinational corporations  such \nas : Ikeja Electric, MTN, IBEDC. These services includes: \na. Petirable Electric Poles Services \nPetirable is dedicated to the production of world -class concrete electric poles. The \norganisation specialises in producing durable, high-quality, and standard concrete electric \npoles designed to meet national and international standards. Their poles, which are SON \n(Standard Organisation of Nigeria) certified  are engineered to withstand Nigeria’s and \nAfrica’s diverse environmental conditions while ensuring safety and reliability in power \ndistribution. \nTheir range of products include:'),
 Document(id='b54fd2cd-2f30-49a2-b478-91975ea8c544', metadata={}, page_content='5 \n \n2.3 PETIRABLE SERVICE

In [79]:
#initialize the chatmodel
chatmodel = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model=chat_model_name,
    temperature=0
)

In [80]:
#using from template method
prompt = ChatPromptTemplate.from_template(
    """You are a student who just completed the student industrial working experience (SIWES) training and serves as a consultant providing insights on siwes training in Nigeria.
You will be provided with the context: {context} to answer the user's question.
The context includes sections about the company where the SIWES was carried out, the activities carried out during the experience, the lessons and tools used in carrying out activities during the training.
Provide a comprehensive response.
Include relevant sources or links from the context in your response at the end of each answer, include a statement: "To read more, check out this link: [insert link]."
Avoid unnecessary or unrelated details. Format the text output clearly and professionally in an HTML format.
question: {question}""")


In [81]:
# question = "What is the name of the company where siwes was undertaken?"

# #retrieve the document
# get_doc = siwes_report_retriever.invoke(question)
# # Prepare the input for the chain
# input = {"context": get_doc, "question": question}

# # Create the chain
# chain = prompt | chatmodel | StrOutputParser()

# answer = chain.invoke(input)
# print(answer)

In [82]:
question = "What services does the organization where the siwes training took place render?"

#retrieve the document
get_doc = siwes_report2_retriever.invoke(question)
# Prepare the input for the chain
input = {"context": get_doc, "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()

answer = chain.invoke(input)
print(answer)

<p><strong>Organization where the SIWES training took place</strong></p>
<p>The SIWES training was carried out at <strong>PETIRABLE ELECTRIC POLES LTD</strong>, KM 2 GBERIGBE – IMOTA ROAD, IKORODU, LAGOS STATE. The report is submitted by Ogungbesan Victor Ayodeji, Department of Mechanical Engineering, Ekiti State University, Ado Ekiti, for the period October – December 2025.</p>
<p>To read more, check out this link: [Document(id='7f27199f-53fc-4bef-bdd3-e9f82088004f')].</p>

<p><strong>Services rendered by the organization</strong></p>
<p>Based on the company profile provided in the SIWES report, PETIRABLE ELECTRIC POLES LTD is positioned in the pole manufacturing and power solutions sector.</p>
<ul>
<li><strong>Pole manufacturing</strong> – production of electric poles as core manufacturing output.</li>
<li><strong>Power solutions</strong> – provision of power-related solutions to clients.</li>
</ul>
<p>The company’s stated vision is:</p>
<blockquote>To become the number one pole manu